In [1]:
from cmath import nan

import numpy as np
import yfinance as yf
import pandas as pd
import datetime as dt

import hist_volatility
from riskfree_and_spot import *
from hist_volatility import *
from blackscholes import *
from binomial import *
from imp_volatility import *


In [2]:

option_style = "american" #european / american
valuation_date = "2026-08-04" #YYYY-MM-DD
ticker = "MSFT"
expiration = "2026-08-05"
strike = 487.5
option_type = "Call" # call / put
volatility_method = "historical" #historical / (implied /not yet implemented/)
hist_vol_lookback = 1 #in years

#BINOMIAL
n_for_binomial = 1500 #t0 + n steps

In [3]:
spot_price = get_spot_price_data(ticker=ticker, valuation_date=valuation_date)
t_t_m = option_tenor_calc(val_date=valuation_date, exp_date=expiration)
risk_free_rate = get_risk_free_rate(val_date=valuation_date, exp_date=expiration)
vol = get_volatility(ticker=ticker, val_date=valuation_date, lookback=hist_vol_lookback)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
#UPDATED_OPTION_CHAIN_QUARY

ticker_obj = yf.Ticker(ticker)
exps = ticker_obj.options
all_data = []

for date in exps:
    option_chain = ticker_obj.option_chain(date)
    if option_type.lower() =="call":
        type_chain = option_chain.calls
    elif option_type.lower() =="put":
        type_chain = option_chain.puts
    else:
        raise ValueError("option_style must be either 'call' or 'put'")
    type_chain["expiration"] = date

    all_data.append(type_chain)

all_data = pd.concat(all_data, ignore_index = True)
all_data


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15


In [5]:
all_data.to_csv("../data/option_data.csv", index=False)

In [6]:
data = pd.read_csv("../data/option_data.csv")


In [7]:
data.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.2,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.9,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.2,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.2,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.9,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07


In [8]:
tenor_list = []

for i, row in data.iterrows():
    exp_date = row["expiration"]
    tenor = option_tenor_calc(val_date=valuation_date, exp_date=exp_date)
    tenor_list.append(tenor)

data["tenor"] = tenor_list


In [9]:
data

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123


In [10]:
exp_list = data["expiration"].unique()

rf_for_exp_dict = {
    exp: get_risk_free_rate(val_date=valuation_date, exp_date=exp)
    for exp in exp_list
}

data["rf_rate"] = data["expiration"].map(rf_for_exp_dict)

data

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219,0.037800
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219,0.037800
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219,0.037800
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219,0.037800
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219,0.037800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123,0.042159
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123,0.042159
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123,0.042159
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123,0.042159


In [11]:
def new_raph_imp_vol(opt_type:str, target:float, spot_p:float, strike:float, T:float, r:float):
    iterations = 200
    precision = 1.0e-5
    vol = 0.1

    for i in range(0, iterations):
        if opt_type.lower() == "call":
            bs = bs_call(S=spot_p, K=strike, T=T, r=r, vol=vol)
        elif opt_type.lower() == "put":
            bs = bs_put(S=spot_p, K=strike, T=T, r=r, vol=vol)
        else:
            raise ValueError("option_style must be either 'call' or 'put'")

        vega = bs_vega(S=spot_p, K=strike, T=T, r=r, vol=vol)
        diff = target - bs

        if abs(diff) < precision:
            return float(vol)

        if vega == 0 or np.isnan(vega) or np.isinf(vega):
            return np.nan

        vol = vol + (diff / vega)

        if vol <= 0 or np.isnan(vol) or np.isinf(vol):
            return np.nan


In [12]:
test = new_raph_imp_vol(opt_type="call", target= 56, spot_p=100, strike=100, T=1, r=0.05)
print(test)

1.5066759904935285


In [13]:
test = find_vol_bs(opt_type="call", target_value=56,S=100,K=100,T=1, r=0.05, vol=0.1)
print(test)

1.5066759904935285


In [14]:
imp_vol_list = []

for i, row in data.iterrows():
    opt_type = option_type.lower()
    target_value = (row["bid"] + row["ask"]) / 2
    s = spot_price
    k = row["strike"]
    t = row["tenor"]
    r = row["rf_rate"]
    vol = new_raph_imp_vol(opt_type=opt_type, target=target_value, spot_p=s, strike=k, T=t, r=r)
    imp_vol_list.append(vol)

data["imp_vol"] = imp_vol_list
data


C:\Users\Balint\AppData\Local\Temp\ipykernel_28668\642515742.py:23: RuntimeWarning: overflow encountered in scalar divide
  vol = vol + (diff / vega)
C:\Users\Balint\PycharmProjects\option_pricing_calculator\Work_In_Progress\imp_volatility.py:14: RuntimeWarning: overflow encountered in scalar power
  d1 = (np.log(S/K) + (r + 0.5 * vol ** 2) * T) / (vol*np.sqrt(T))
C:\Users\Balint\PycharmProjects\option_pricing_calculator\Work_In_Progress\imp_volatility.py:24: RuntimeWarning: overflow encountered in scalar power
  d1 = (np.log(S/K) + (r + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,imp_vol
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344011
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.345017
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344173
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.343435


In [15]:
data.to_csv("../data/option_data_calc.csv", index=False)